In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree


class NDealkylation(MorphingOperator):
    def __init__(self):
        super(NDealkylation, self).__init__()
        self._name = "N-Dealkylation (Phase I - Advanced)"
        self._target_bonds = []
        self.PATTERN = Chem.MolFromSmarts("[NX3;H0,H1;!a;!$(N-C=O)][CX4;H1,H2,H3]")

    def setOriginal(self, mol):
        super(NDealkylation, self).setOriginal(mol)
        self._target_bonds = []
        
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_bonds.append((match[0], match[1]))

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        
        if not self._target_bonds:
            return MolpherMol(other=rdkit_mol)
            
        idx_n, idx_c = random.choice(self._target_bonds)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            
            bond = rw_mol.GetBondBetweenAtoms(idx_n, idx_c)
            if bond:
                rw_mol.RemoveBond(idx_n, idx_c)
                
            new_mol = rw_mol.GetMol()
            
            atom_n = new_mol.GetAtomWithIdx(idx_n)
            atom_n.SetNoImplicit(False)
            atom_n.SetNumExplicitHs(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            
            frags_mols = Chem.GetMolFrags(new_mol, asMols=True)
            
            if frags_mols:
                
                frags_indices = Chem.GetMolFrags(new_mol, asMols=False)
                frags_with_meta = list(zip(frags_mols, frags_indices))
                frags_with_meta = sorted(
                    frags_with_meta, 
                    key=lambda x: (idx_n in x[1], x[0].GetNumAtoms()), 
                    reverse=True
                )
                
                final_mol = frags_with_meta[0][0]
                
                Chem.AssignStereochemistry(final_mol, cleanIt=True, force=True)
                clean_smiles = Chem.MolToSmiles(final_mol)
                return MolpherMol(clean_smiles)
                
            return MolpherMol(other=rdkit_mol)
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


class NitrogenSulfation(MorphingOperator):
    def __init__(self):
        super(NitrogenSulfation, self).__init__()
        self._name = "N-Sulfation (Amines 1°, 2°, 3° & Aromatic)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[N;X3;!$(N-C=O);!$(N-C(=O)N);!$(N-C(=O)O)]")

    def setOriginal(self, mol):
        super(NitrogenSulfation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            self._target_atoms.append(match[0])

    def morph(self):
        if not self.original: return None
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return None
        
        if not self._target_atoms:
            return MolpherMol(other=rdkit_mol)
        
        nitrogen_idx = random.choice(self._target_atoms)
        
        try:
            rw_mol = Chem.RWMol(rdkit_mol)
            is_tertiary = (rw_mol.GetAtomWithIdx(nitrogen_idx).GetTotalNumHs() == 0)
            
            sulfur_idx = rw_mol.AddAtom(Chem.Atom(16)) # S
            rw_mol.AddBond(nitrogen_idx, sulfur_idx, Chem.BondType.SINGLE)
            
            o1_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o1_idx, Chem.BondType.DOUBLE)
            
            o2_idx = rw_mol.AddAtom(Chem.Atom(8)) # =O
            rw_mol.AddBond(sulfur_idx, o2_idx, Chem.BondType.DOUBLE)
            
            o3_idx = rw_mol.AddAtom(Chem.Atom(8)) # -OH
            rw_mol.AddBond(sulfur_idx, o3_idx, Chem.BondType.SINGLE)
            
            new_mol = rw_mol.GetMol()
            
            for idx in [o1_idx, o2_idx, o3_idx]:
                atom = new_mol.GetAtomWithIdx(idx)
                atom.SetNoImplicit(False)
                atom.SetNumExplicitHs(0)
                atom.SetFormalCharge(0)
            
            n_atom = new_mol.GetAtomWithIdx(nitrogen_idx)
            n_atom.SetNoImplicit(False)
            n_atom.SetNumExplicitHs(0)
            
            if is_tertiary:
                
                n_atom.SetFormalCharge(1)
            else:
                
                n_atom.SetFormalCharge(0)
                
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name
    
dealk_op = NDealkylation()
nitrogen_sulfation = NitrogenSulfation()

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
        
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target
closest_info = FindClosest()

# Source: Tramadol
start_mol  = MolpherMol("CN(C)C[C@H]1CCCC[C@@]1(c1cccc(OC)c1)O")
# Target: N-desmethyl-N-sulfo-tramadol
# N(CH3)2 → NH(CH3) → N(CH3)(SO3H)
target_mol = MolpherMol("CN(S(=O)(=O)O)C[C@H]1CCCC[C@@]1(c1cccc(OC)c1)O")
tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = (nitrogen_sulfation, dealk_op)

print("--- STARTING MOLPHER SEARCH TREE ---")
max_generations = 40
while not tree.path_found and tree.generation_count < max_generations:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
    
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    if closest_info.closest_mol:
        print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)    

--- STARTING MOLPHER SEARCH TREE ---
Generation #1
Molecules in tree: 4
Closest to target: COC1=CC(C2(O)CCCCC2C[N+](C)(C)S(=O)(=O)O)=CC=C1 (Distance: 0.3036)
----------------------------------------
Generation #2
Molecules in tree: 8
Closest to target: COC1=CC(C2(O)CCCCC2CN(C)S(=O)(=O)O)=CC=C1 (Distance: 0.0000)
----------------------------------------
